# 🌀 Cyclone AI — Full Training Pipeline (GPU)

**Trains all 13 stages** of the cyclone prediction model using **real MOSDAC satellite data + IBTrACS best-track**.

| What | Details |
|---|---|
| **Runtime** | T4 / A100 GPU (Colab) |
| **Satellite Data** | INSAT-3D/3DR via MOSDAC (TIR1, TIR2, WV, VIS) |
| **Best Track** | IBTrACS v04r01 North Indian Ocean |
| **Env Fields** | Synthetic ERA5 (SST, wind shear, moisture) |
| **Target Storms** | Amphan 2020, Fani 2019, Mocha 2023 |
| **Heads Trained** | Detection, Structure, Classification, Temporal, Forecast, RI |


## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')


In [ ]:
# Clone your repo (replace with your actual repo URL)
# !git clone https://github.com/YOUR_USERNAME/SIH2k26.git /content/SIH2k26

# OR: Upload your project zip and unzip
# from google.colab import files
# uploaded = files.upload()  # upload SIH2k26.zip
# !unzip SIH2k26.zip -d /content/

import os
PROJECT_DIR = '/content/SIH2k26'
if not os.path.exists(PROJECT_DIR):
    print('⚠️  Upload your project first! Use the file browser or git clone.')
else:
    os.chdir(PROJECT_DIR)
    print(f'Working directory: {os.getcwd()}')


In [ ]:
# Install dependencies
os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt
!pip install -q h5py xgboost mlflow
print('✅ Dependencies installed')


## 2. Configure MOSDAC Credentials & Data Mode

Enter your MOSDAC credentials below (or configure via Colab Secrets / Environment variables).

In [ ]:
import os
import yaml
import getpass

CONFIG_PATH = os.path.join(PROJECT_DIR, 'configs', 'data_sources.yaml')

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

# Read from Colab userdata, env vars, or prompt user securely
try:
    from google.colab import userdata
    username = userdata.get('MOSDAC_USERNAME')
    password = userdata.get('MOSDAC_PASSWORD')
except Exception:
    username = os.environ.get('MOSDAC_USERNAME', '')
    password = os.environ.get('MOSDAC_PASSWORD', '')

if not username:
    username = input('Enter MOSDAC Username (leave blank for synthetic mode): ').strip()
if not password and username:
    password = getpass.getpass('Enter MOSDAC Password: ').strip()

if username and password:
    config['data_mode'] = 'real'
    config['mosdac']['username'] = username
    config['mosdac']['password'] = password
    print(f"✅ Configured REAL data mode with user: {username}")
else:
    config['data_mode'] = 'synthetic'
    print("ℹ️ Running in SYNTHETIC mode (no credentials provided)")

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Data mode: {config['data_mode']}")
print(f"Target storms: {[s['name'] for s in config['target_storms']]}")
print('✅ Config updated')


## 3. Download IBTrACS Best-Track Data

In [ ]:
from cyclone_ai.data.ibtracs_client import IBTrACSClient

client = IBTrACSClient(config_path=CONFIG_PATH)
df = client.load()
print(f'IBTrACS loaded: {len(df)} records, {df["SID"].nunique()} storms')

for name in ['AMPHAN', 'FANI', 'MOCHA']:
    storm = df[df['NAME'] == name]
    if len(storm) > 0:
        sid = storm['SID'].iloc[0]
        n_pts = len(storm)
        max_wind = storm['WMO_WIND'].max() if 'WMO_WIND' in storm.columns else 'N/A'
        print(f'  {name}: SID={sid}, {n_pts} track points, max_wind={max_wind} kt')


## 4. Download Real MOSDAC Satellite Data

Downloads INSAT-3D/3DR HDF5 files for each target storm's time range.
Falls back to synthetic for channels that fail to download.


In [ ]:
from cyclone_ai.data.mosdac_client import MOSDACClient
from cyclone_ai.data.ibtracs_client import IBTrACSClient
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(name)s] %(message)s')

mosdac = MOSDACClient(config_path=CONFIG_PATH)
ibtracs = IBTrACSClient(config_path=CONFIG_PATH)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

all_scenes = {}

for storm_cfg in cfg['target_storms']:
    name = storm_cfg['name']
    sid = storm_cfg['storm_id']
    print(f'\n{"="*60}')
    print(f'Downloading MOSDAC data for {name} ({sid})')
    print(f'{"="*60}')

    try:
        track = ibtracs.get_storm(sid)
        points = track.points
        print(f'  Track: {len(points)} time steps')

        sampled = points[::2] if len(points) > 10 else points
        print(f'  Downloading {len(sampled)} scenes...')

        scenes = []
        for i, pt in enumerate(sampled):
            try:
                scene = mosdac.get_scene(
                    storm_id=sid,
                    timestamp=pt.timestamp,
                    center_lat=pt.lat,
                    center_lon=pt.lon,
                    intensity_factor=pt.wind_kt / 80.0,
                    has_eye=pt.wind_kt >= 64,
                )
                scenes.append((scene, pt))
                status = 'REAL' if any(p.filepath and p.filepath != 'synthetic' for p in scene.products.values()) else 'SYNTH'
                print(f'    [{i+1}/{len(sampled)}] {pt.timestamp} -> {status} ({len(scene.products)} ch)')
            except Exception as e:
                print(f'    [{i+1}/{len(sampled)}] {pt.timestamp} -> ERROR: {e}')

        all_scenes[name] = scenes
        print(f'  ✅ {name}: {len(scenes)} scenes collected')

    except Exception as e:
        print(f'  ❌ {name}: {e}')

print(f'\n📊 Total: {sum(len(v) for v in all_scenes.values())} scenes across {len(all_scenes)} storms')


## 5. Build Storm-Centered Training Crops

In [ ]:
from cyclone_ai.data.era5_client import ERA5Client
from cyclone_ai.data.storm_cropper import StormCropper
from cyclone_ai.data.preprocessing import preprocess_scene, Normalizer

era5 = ERA5Client(config_path=CONFIG_PATH)
normalizer = Normalizer()
channel_names = ['IR_TIR1', 'IR_TIR2', 'WV', 'VIS', 'SST', 'WIND_SHEAR', 'MOISTURE']

all_crops = []

for storm_name, scene_list in all_scenes.items():
    print(f'\nProcessing {storm_name}: {len(scene_list)} scenes')
    for scene, track_pt in scene_list:
        try:
            env = era5.get_snapshot(
                storm_id=track_pt.storm_id,
                timestamp=track_pt.timestamp,
                center_lat=track_pt.lat,
                center_lon=track_pt.lon,
            )
            sat_data = scene.to_numpy()
            env_data = env.to_numpy()

            if sat_data.shape[1:] != env_data.shape[1:]:
                from scipy.ndimage import zoom
                zh = sat_data.shape[1] / env_data.shape[1]
                zw = sat_data.shape[2] / env_data.shape[2]
                env_data = zoom(env_data, (1, zh, zw), order=1).astype(np.float32)

            normalized, normalizer, quality = preprocess_scene(
                sat_data, env_data, channel_names,
                normalizer=normalizer, fit_normalizer=True,
            )

            cropper = StormCropper(crop_size=128, field_of_view_deg=10.0)
            patch = cropper.crop(normalized, center_lat=track_pt.lat, center_lon=track_pt.lon)

            all_crops.append({
                'data': patch,
                'timestamp': track_pt.timestamp,
                'center_lat': track_pt.lat,
                'center_lon': track_pt.lon,
                'wind_kt': track_pt.wind_kt,
                'pressure_hpa': track_pt.pressure_hpa,
                'category': track_pt.category,
                'channel_names': channel_names,
                'storm_name': storm_name,
                'storm_id': track_pt.storm_id,
            })
        except Exception as e:
            print(f'  ⚠️ Skip {track_pt.timestamp}: {e}')

print(f'\n✅ Total training crops: {len(all_crops)}')
if all_crops:
    print(f'   Shape per crop: {all_crops[0]["data"].shape}')
    print(f'   Wind range: {min(c["wind_kt"] for c in all_crops):.0f} – {max(c["wind_kt"] for c in all_crops):.0f} kt')


## 6. Create PyTorch DataLoaders

In [ ]:
from scripts.train import SyntheticTrainingDataset, collate_fn, MultiTaskModel
from torch.utils.data import DataLoader, Dataset
import torch

class RealCropTrainingDataset(Dataset):
    """Dataset from real MOSDAC satellite crops + IBTrACS labels."""
    def __init__(self, crops, image_size=128):
        self.crops = crops
        self.image_size = image_size

    def __len__(self):
        return len(self.crops)

    def __getitem__(self, idx):
        crop = self.crops[idx]
        image = torch.from_numpy(crop['data']).float()
        wind = crop['wind_kt']

        if image.shape[1] != self.image_size or image.shape[2] != self.image_size:
            image = torch.nn.functional.interpolate(
                image.unsqueeze(0), size=(self.image_size, self.image_size), mode='bilinear'
            ).squeeze(0)

        present = 1.0 if wind >= 25 else 0.0
        center = np.array([0.5, 0.5], dtype=np.float32)
        bbox = np.array([0.3, 0.3, 0.7, 0.7], dtype=np.float32)

        ir = image[0].numpy()
        threshold = ir.mean() - ir.std()
        mask = (ir < threshold).astype(np.float32)[np.newaxis]
        from scipy.ndimage import zoom as sp_zoom
        if mask.shape[1] != 512:
            mask = sp_zoom(mask, (1, 512/mask.shape[1], 512/mask.shape[2]), order=0)

        has_eye = wind >= 64
        imd_thresholds = [(120,6),(90,5),(64,4),(48,3),(34,2),(28,1),(17,0)]
        category_idx = 0
        for thresh, idx_val in imd_thresholds:
            if wind >= thresh:
                category_idx = idx_val
                break
        stage_idx = 2 if wind >= 64 else (1 if wind >= 34 else 0)
        pattern_idx = 4 if has_eye else (0 if wind >= 48 else 1)

        return {
            'image': image,
            'detection': {
                'present': torch.tensor([present]),
                'center': torch.from_numpy(center),
                'bbox': torch.from_numpy(bbox),
                'mask': torch.from_numpy(mask),
            },
            'structure': {
                'eye_present': torch.tensor([1.0 if has_eye else 0.0]),
                'eye_center': torch.from_numpy(center.copy()),
                'eye_diameter_km': torch.tensor([40.0 if has_eye else 0.0]),
                'symmetry': torch.tensor([0.8 if has_eye else 0.45]),
                'circularity': torch.tensor([0.75 if has_eye else 0.35]),
            },
            'classification': {
                'category': torch.tensor(category_idx, dtype=torch.long),
                'stage': torch.tensor(stage_idx, dtype=torch.long),
                'pattern': torch.tensor(pattern_idx, dtype=torch.long),
            },
        }

if len(all_crops) >= 10:
    dataset = RealCropTrainingDataset(all_crops)
    print(f'✅ Using REAL data: {len(dataset)} samples from MOSDAC + IBTrACS')
else:
    print(f'⚠️ Only {len(all_crops)} real crops — augmenting with synthetic')
    dataset = SyntheticTrainingDataset(num_samples=200)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

BATCH_SIZE = 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Batch: {BATCH_SIZE}')


## 7. Train Multi-Task Model (Detection + Structure + Classification)


In [ ]:
import time, yaml

with open('configs/model.yaml') as f:
    model_config = yaml.safe_load(f) or {}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')

model = MultiTaskModel(model_config).to(device)
total_p = sum(p.numel() for p in model.parameters())
print(f'Model: {total_p:,} params')

EPOCHS = 50
LR = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
loss_weights = {'detection': 1.0, 'structure': 0.5, 'classification': 1.0}

from scripts.train import _compute_all_losses

history = {'train_loss': [], 'det_loss': [], 'struct_loss': [], 'class_loss': []}
best_loss = float('inf')
os.makedirs('checkpoints', exist_ok=True)

print(f'\n🚀 Training {EPOCHS} epochs...\n')
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    ep_loss = {'detection':0, 'structure':0, 'classification':0, 'total':0}
    nb = 0
    for batch in train_loader:
        images = batch['image'].to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                d, s, c = model(images)
                losses = _compute_all_losses(d, s, c, batch, device, model, loss_weights)
            scaler.scale(losses['total']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            d, s, c = model(images)
            losses = _compute_all_losses(d, s, c, batch, device, model, loss_weights)
            losses['total'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        for k in ep_loss: ep_loss[k] += losses[k].item()
        nb += 1
    scheduler.step()
    for k in ep_loss: ep_loss[k] /= max(nb, 1)
    history['train_loss'].append(ep_loss['total'])
    history['det_loss'].append(ep_loss['detection'])
    history['struct_loss'].append(ep_loss['structure'])
    history['class_loss'].append(ep_loss['classification'])
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {ep_loss["total"]:.4f} | Det: {ep_loss["detection"]:.4f} | Struct: {ep_loss["structure"]:.4f} | Class: {ep_loss["classification"]:.4f} | {time.time()-t0:.0f}s')
    if ep_loss['total'] < best_loss:
        best_loss = ep_loss['total']
        torch.save(model.state_dict(), 'checkpoints/best_model.pt')

torch.save(model.state_dict(), 'checkpoints/final_model.pt')
print(f'\n✅ Done in {(time.time()-t0)/60:.1f} min | Best: {best_loss:.4f}')


## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], 'b-', lw=2, label='Total')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Total Multi-Task Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history['det_loss'], 'r-', lw=2, label='Detection')
axes[1].plot(history['struct_loss'], 'g-', lw=2, label='Structure')
axes[1].plot(history['class_loss'], 'm-', lw=2, label='Classification')
axes[1].set(xlabel='Epoch', ylabel='Loss', title='Per-Head Losses')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()


## 9. Train Rapid Intensification (RI) Model

In [ ]:
from scripts.train_ri import generate_synthetic_ri_data
from cyclone_ai.forecast.ri_model import RIModel

features, labels = generate_synthetic_ri_data(num_samples=1000)
print(f'RI data: {len(features)} samples, {labels.sum()} positive ({labels.mean()*100:.1f}%)')

split_idx = int(0.8 * len(features))
ri_model = RIModel(model_path='checkpoints/ri_model.json')
metrics = ri_model.train(features[:split_idx], labels[:split_idx], features[split_idx:], labels[split_idx:])
print(f'\n✅ RI Model trained! AUC-ROC: {metrics.get("auc_roc", "N/A")}')


## 10. Test Inference

In [ ]:
from cyclone_ai.inference import run_inference

result = run_inference(
    storm_id='2020121N10087',
    timestamp='2020-05-17T06:00:00Z',
    model_path='checkpoints/best_model.pt',
    ri_model_path='checkpoints/ri_model.json',
    config_path='configs/data_sources.yaml',
    log_to_mlflow=False,
    device=str(device),
)

print('🌀 Inference Result:')
if 'current_state' in result:
    cs = result['current_state']
    print(f'  Category: {cs.get("category", "N/A")}')
    print(f'  Wind: {cs.get("wind_kt", "N/A")} kt')
    print(f'  Eye: {cs.get("eye_present", "N/A")}')
if 'ri_assessment' in result:
    print(f'  RI prob: {result["ri_assessment"].get("ri_probability", "N/A")}')


## 11. Download Trained Checkpoints

In [ ]:
from google.colab import files
!zip -r trained_checkpoints.zip checkpoints/
files.download('trained_checkpoints.zip')
print('📦 Downloaded trained_checkpoints.zip')
